[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/18_warehouses-dbt.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 18 of 21*

# Warehouses & dbt

Modern analytics engineering: how cloud warehouses store and bill, clustering and distribution keys, views vs tables vs materialized views, and dbt's models, tests, incremental loads and snapshots. You'll build a tiny dbt.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A warehouse charges for what you read or for the time you run. Design around whichever one your bill is made of.

**What interviewers really test here:** Partitioning and clustering, materializations, incremental models and controlling cost.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. How a cloud warehouse works *(Basics)*
- [ ] 2. Physical design: sort and distribution *(Core)*
- [ ] 3. dbt: software engineering for SQL *(Core)*
- [ ] 4. Incremental models, snapshots and cost control *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### dbt materializations

| Type | Builds | Use for |
| --- | --- | --- |
| view | a view | staging, light logic |
| table | a full table each run | small or medium marts |
| incremental | adds or merges new rows | large event tables |
| ephemeral | inlined as a CTE | small reusable steps |

---
# Part 2: Lessons, step by step

## Lesson 1: How a cloud warehouse works

*Level: Basics*

Snowflake, BigQuery, Redshift and Databricks SQL differ in details, but share these ideas:



- **Columnar storage** with compression and min/max per block.
- **Massively parallel:** a query is split across many workers.
- **Storage separate from compute,** in most modern designs, so each scales alone.



| Warehouse | You pay for | Optimise by |
| --- | --- | --- |
| BigQuery (on-demand) | bytes scanned | partitioning, clustering, fewer columns |
| Snowflake | running time × warehouse size | auto-suspend, right size |
| Redshift | nodes per hour, or serverless | sort and distribution keys |



**Scanning cost.** A 10 TB table at a made-up 5 per TB costs 50 for a full scan. Using 10% of columns costs 5. Filtering to 5% of partitions as well costs **0.25**. `SELECT *` on a big table can be a very expensive habit.



**Sizing compute.** Each size up doubles the price per hour. If it also halves the time, the cost doesn't change, and you simply get answers faster: 8 hours at size 1 costs 8, and 2 hours at size 4 also costs 8. If scaling is imperfect, at 1.5 times faster per step, the same jump costs 14.2.



**Auto-suspend.** A warehouse busy 4 hours a day bills 24 hours without suspending. With a 6-minute idle timeout across 10 jobs, it bills 5 hours: 79% less.



**Result caches** return identical queries for free, until the data changes.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q16](#q16), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 2: Physical design: sort and distribution

*Level: Core*

**Clustering or sort keys** keep similar values together, so min/max per block can skip most data. Filtering 12–25 across blocks 1–10, 11–20, 21–30 and 31–40 reads only 2 of them. Choose columns people filter on, starting with date.



**Distribution keys (Redshift-style).** Choose how rows spread across nodes:



| Style | How | Good for |
| --- | --- | --- |
| Key | hash of a column | tables often joined on it, as no shuffle's needed |
| Even | round robin | no clear join key |
| All | a copy on each node | small dimensions |



A bad key causes skew. In 10 rows with 8 from "in", distributing by country on 2 nodes gives a skew of 1.6, while user id gives 1.0.



**Concurrency.** Warehouses limit how many queries run at once. Durations 10, 10, 10, 5 with 2 slots give waits of 0, 0, 10, 10, so the fourth query waits 10 seconds to do 5 seconds of work. Separate heavy jobs from dashboards.



**View, table or materialized view?**



| Type | Pays at | Fresh? | Use when |
| --- | --- | --- | --- |
| View | each read | always | few reads, or cheap logic |
| Table | each rebuild | as of the last build | many reads |
| Materialized view | on refresh | managed | supported, simple aggregations |



A query costing 10, read 100 times and rebuilt once a day: a view costs 1,000, and a table costs 10. Read once, with hourly rebuilds, a view costs 10 and a table 240.

**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q17](#q17), [Q19](#q19), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 3: dbt: software engineering for SQL

*Level: Core*

A warehouse full of unversioned SQL scripts soon becomes unmaintainable. **dbt** treats SQL as code: each model is a SELECT in a file, in Git, with tests and documentation.



```python
-- models/staging/stg_orders.sql
SELECT DISTINCT id, day, amount
FROM {{ source('shop', 'raw_orders') }}
WHERE status = 'paid'

-- models/marts/daily_revenue.sql
SELECT day, SUM(amount) AS total
FROM {{ ref('stg_orders') }}
GROUP BY day
```



`ref()` is the key idea. dbt reads these references to build a graph, so it knows that stg\_orders must run first. Questions here rebuild that logic: finding refs, ordering models and compiling them to real SQL.



| Layer | Purpose |
| --- | --- |
| Staging | one clean model per source: rename, cast, dedupe |
| Intermediate | reusable joins and logic |
| Marts | tables for the business |



In the practice data, order 3 appears twice and order 2 failed. Staging removes both issues, so the mart can't double count: d2's revenue is 7, not 14.



**Selecting what to run:** `b+` means b and everything downstream; `+c` means c and everything upstream.



**Tests** compile to queries that should return no rows, such as unique\_id and not\_null\_id. Run them in CI, on a small sample, with every pull request.



**Your QA experience applies directly here.**

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q10](#q10), [Q2](#q2), [Q3](#q3), [Q11](#q11), [Q14](#q14), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 4: Incremental models, snapshots and cost control

*Level: Advanced*

**Incremental models** process only new data. Reprocessing recent days catches late rows. With existing {d1: 10, d2: 5} and new rows d2: 3, d2: 4, d3: 1, d1: 99, cut-off d2 gives {d1: 10, d2: 7, d3: 1}. The old d1 row's ignored, and that's the trade-off: very late data needs a full refresh.



**Snapshots** build SCD 2 from a source that only holds current values. Each run compares and closes changed rows: b→B closes row 2 and opens a new version; a new id 3 opens a row. Without snapshots, you can't get that history back later.



**One definition of each metric.** Sales counts paid + shipped as 15, and finance counts only paid as 10. Now two teams argue in a meeting over "revenue". A **semantic layer** defines metrics once, in code, and every tool uses that definition. Often the hardest part is getting people to agree, not the technology.



**Control costs by measuring them.** Tag queries by team and report it: team a used 3 credits, and team b 2. Normalise queries to find repeated shapes: two queries that differ only in id are one shape costing 200, more than a one-off at 150.



1. Find the top 10 most expensive query shapes.
2. Partition, cluster, and stop using SELECT \*.
3. Materialise heavy shared logic.
4. Set auto-suspend and budgets.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q20](#q20), [Q7](#q7), [Q8](#q8). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re

import sqlite3

SCHEMA = """
CREATE TABLE raw_orders (id INT, day TEXT, status TEXT, amount INT);
INSERT INTO raw_orders VALUES (1,'d1','paid',10),(2,'d1','failed',5),(3,'d2','paid',7),(3,'d2','paid',7),(4,'d3','paid',1);
"""

def run_sql(query):
    """Run statements on a fresh database and return the last query's rows as lists."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    *setup, last = [s for s in query.split(";") if s.strip()]
    for s in setup:
        con.execute(s)
    rows = con.execute(last).fetchall()
    con.close()
    return [list(r) for r in rows]

from collections import defaultdict

import heapq


<a id="q1"></a>
### Q1. Find ref() Dependencies

🟢 Warm-up · *How dbt builds its graph*

Return the model names used in {{ ref('x') }} in a SQL string, sorted and unique.

**Examples**

- `sql = "SELECT * FROM {{ ref('a') }} JOIN {{ ref( 'b' ) }} USING (id) JOIN {{ ref('a') }} x"` → `['a', 'b']`

In [ ]:
def find_refs(sql):
    """
    Return the model names used in {{ ref('x') }} in a SQL string, sorted and
    unique.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_refs, [
    (("SELECT * FROM {{ ref('a') }} JOIN {{ ref( 'b' ) }} USING (id) JOIN {{ ref('a') }} x",), ['a', 'b']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on 'ref(' and clean each piece.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Split on 'ref(' and clean each piece.

```python
def find_refs(sql):
    out = []
    for piece in sql.split("ref(")[1:]:
        name = piece.split(")")[0].strip().strip("'\"")
        if name not in out:
            out.append(name)
    return sorted(out)
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression.

```python
def find_refs(sql):
    return sorted(set(re.findall(r"ref\(\s*['\"](\w+)['\"]\s*\)", sql)))
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on 'ref(' and clean each piece. That costs O(n) time and O(r) space."

"I can do better: A regular expression. That's O(n) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/find-ref-dependencies)

<a id="q2"></a>
### Q2. Compile a Model

🟢 Warm-up · *Jinja to SQL*

Replace {{ ref('x') }} with schema.x.

**Examples**

- `sql = "SELECT * FROM {{ ref('orders') }}", schema = 'prod'` → `'SELECT * FROM prod.orders'`

In [ ]:
def compile_model(sql, schema):
    """
    Replace {{ ref('x') }} with schema.x.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compile_model, [
    (("SELECT * FROM {{ ref('orders') }}", 'prod'), 'SELECT * FROM prod.orders'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

String methods in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`re.sub`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** String methods in a loop.

```python
def compile_model(sql, schema):
    while "{{ ref('" in sql:
        start = sql.index("{{ ref('")
        end = sql.index("}}", start) + 2
        name = sql[start + 8:end - 5]
        sql = sql[:start] + schema + "." + name + sql[end:]
    return sql
```

**Cost:** O(n·r) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `re.sub`.

```python
def compile_model(sql, schema):
    return re.sub(r"\{\{\s*ref\('(\w+)'\)\s*\}\}", lambda m: f"{schema}.{m.group(1)}", sql)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: String methods in a loop. That costs O(n·r) time and O(n) space."

"I can do better: `re.sub`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/compile-a-model)

<a id="q3"></a>
### Q3. A Staging Model

🟢 Warm-up · *One clean version of each source*

From raw\_orders, return unique orders that are paid: (id, day, amount), by id.

**Examples**

- `` → `[[1, 'd1', 10], [3, 'd2', 7], [4, 'd3', 1]]`

In [ ]:
def stg_orders():
    """
    From raw_orders, return unique orders that are paid: (id, day, amount), by
    id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stg_orders, [
    ((), [[1, 'd1', 10], [3, 'd2', 7], [4, 'd3', 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Clean up in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DISTINCT and a WHERE in a view, which is what a staging model is.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Clean up in Python.

```python
def stg_orders():
    rows = run_sql("SELECT id, day, status, amount FROM raw_orders")
    out = {}
    for i, d, s, a in rows:
        if s == "paid":
            out[i] = [i, d, a]
    return [out[k] for k in sorted(out)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** DISTINCT and a WHERE in a view, which is what a staging model is.

```python
def stg_orders():
    return run_sql("""CREATE VIEW stg AS SELECT DISTINCT id, day, amount FROM raw_orders WHERE status = 'paid';
        SELECT * FROM stg ORDER BY id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Clean up in Python. That costs O(n log n) time and O(n) space."

"I can do better: DISTINCT and a WHERE in a view, which is what a staging model is. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/a-staging-model)

<a id="q4"></a>
### Q4. Pay per Byte Scanned

🟢 Warm-up · *On-demand pricing*

With a made-up price per TB scanned, return query cost for full scan, column pruned and partition pruned too, rounded to 2 decimals.

**Examples**

- `tb = 10, col_share = 0.1, part_share = 0.05, price = 5` → `(50, 5.0, 0.25)`

In [ ]:
def scan_cost(tb, col_share, part_share, price):
    """
    With a made-up price per TB scanned, return query cost for full scan, column
    pruned and partition pruned too, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scan_cost, [
    ((10, 0.1, 0.05, 5), (50, 5.0, 0.25)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def scan_cost(tb, col_share, part_share, price):
    full = tb * price
    columns = full * col_share
    both = columns * part_share
    return (round(full, 2), round(columns, 2), round(both, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def scan_cost(tb, col_share, part_share, price):
    f = tb * price
    return (round(f, 2), round(f * col_share, 2), round(f * col_share * part_share, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/pay-per-byte-scanned)

<a id="q5"></a>
### Q5. Auto-Suspend Saves Money

🟢 Warm-up · *Idle compute costs*

A warehouse runs jobs for busy hours a day. Without auto-suspend, it bills 24 hours; with it, busy + idle\_after × jobs. Return both and the percent saved, rounded to 1 decimal.

**Examples**

- `busy = 4, idle_after = 0.1, jobs = 10` → `(24, 5.0, 79.2)`

In [ ]:
def auto_suspend(busy, idle_after, jobs):
    """
    A warehouse runs jobs for busy hours a day. Without auto-suspend, it bills
    24 hours; with it, busy + idle_after × jobs. Return both and the percent
    saved, rounded to 1 decimal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(auto_suspend, [
    ((4, 0.1, 10), (24, 5.0, 79.2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def auto_suspend(busy, idle_after, jobs):
    with_it = busy + idle_after * jobs
    saved = (24 - with_it) / 24 * 100
    return (24, round(with_it, 1), round(saved, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def auto_suspend(busy, idle_after, jobs):
    w = busy + idle_after * jobs
    return (24, round(w, 1), round((1 - w / 24) * 100, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/auto-suspend-saves-money)

<a id="q6"></a>
### Q6. Result Cache

🟢 Warm-up · *Same query, no compute*

Warehouses can return a saved result if the query and data haven't changed. Given queries as (text, table\_version), return (hits, compute runs).

**Examples**

- `queries = [('q1', 1), ('q1', 1), ('q2', 1), ('q1', 2)]` → `(1, 3)`

In [ ]:
def result_cache(queries):
    """
    Warehouses can return a saved result if the query and data haven't changed.
    Given queries as (text, table_version), return (hits, compute runs).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(result_cache, [
    (([('q1', 1), ('q1', 1), ('q2', 1), ('q1', 2)],), (1, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of seen pairs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of seen pairs.

```python
def result_cache(queries):
    seen, hits = [], 0
    for q in queries:
        if q in seen:
            hits += 1
        else:
            seen.append(q)
    return (hits, len(seen))
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set.

```python
def result_cache(queries):
    n = len(set(queries))
    return (len(queries) - n, n)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of seen pairs. That costs O(n²) time and O(n) space."

"I can do better: A set. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/result-cache)

<a id="q7"></a>
### Q7. Cost by Team

🟢 Warm-up · *Make spending visible*

Given query records (team, seconds, credits\_per\_hour), return credits per team rounded to 2 decimals, largest first.

**Examples**

- `records = [('a', 3600, 2), ('b', 1800, 4), ('a', 1800, 2)]` → `[['a', 3.0], ['b', 2.0]]`

In [ ]:
def cost_by_team(records):
    """
    Given query records (team, seconds, credits_per_hour), return credits per
    team rounded to 2 decimals, largest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cost_by_team, [
    (([('a', 3600, 2), ('b', 1800, 4), ('a', 1800, 2)],), [['a', 3.0], ['b', 2.0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict` and sorted.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary.

```python
def cost_by_team(records):
    totals = {}
    for team, seconds, rate in records:
        totals[team] = totals.get(team, 0) + seconds / 3600 * rate
    return [[t, round(v, 2)] for t, v in sorted(totals.items(), key=lambda x: -x[1])]
```

**Cost:** O(n log t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict` and sorted.

```python
def cost_by_team(records):
    totals = defaultdict(float)
    for t, s, r in records:
        totals[t] += s * r / 3600
    return sorted(([t, round(v, 2)] for t, v in totals.items()), key=lambda x: -x[1])
```

**Cost:** O(n log t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary. That costs O(n log t) time and O(t) space."

"I can do better: `defaultdict` and sorted. That's O(n log t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/cost-by-team)

<a id="q8"></a>
### Q8. Most Expensive Queries

🟢 Warm-up · *Where to start tuning*

Normalise queries by replacing numbers with ?, group the same shape, and return the top k by total bytes.

**Examples**

- `queries = [('SELECT * FROM t WHERE id=1', 100), ('SELECT * FROM t WHERE id=2', 100), ('SELECT 1 FROM u', 150)], k = 1` → `[['SELECT * FROM t WHERE id=?', 200]]`

In [ ]:
def top_shapes(queries, k):
    """
    Normalise queries by replacing numbers with ?, group the same shape, and
    return the top k by total bytes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_shapes, [
    (([('SELECT * FROM t WHERE id=1', 100), ('SELECT * FROM t WHERE id=2', 100), ('SELECT 1 FROM u', 150)], 1), [['SELECT * FROM t WHERE id=?', 200]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop over characters to normalise.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression and a `Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop over characters to normalise.

```python
def top_shapes(queries, k):
    totals = {}
    for q, b in queries:
        shape = re.sub(r"\d+", "?", q)
        totals[shape] = totals.get(shape, 0) + b
    return sorted(([s, v] for s, v in totals.items()), key=lambda x: -x[1])[:k]
```

**Cost:** O(total) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression and a `Counter`.

```python
def top_shapes(queries, k):
    from collections import Counter
    c = Counter()
    for q, b in queries:
        c[re.sub(r"\d+", "?", q)] += b
    return [[s, v] for s, v in c.most_common(k)]
```

**Cost:** O(total) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop over characters to normalise. That costs O(total) time and O(s) space."

"I can do better: A regular expression and a `Counter`. That's O(total) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/most-expensive-queries)

<a id="q9"></a>
### Q9. Generate dbt Tests

🟢 Warm-up · *Tests from a config*

Given columns with tests, return the test names as 'test\_column', sorted. dbt turns each into a SQL query that should return no rows.

**Examples**

- `config = {'id': ['unique', 'not_null'], 'status': ['accepted_values']}` → `['accepted_values_status', 'not_null_id', 'unique_id']`

In [ ]:
def test_names(config):
    """
    Given columns with tests, return the test names as 'test_column', sorted.
    dbt turns each into a SQL query that should return no rows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(test_names, [
    (({'id': ['unique', 'not_null'], 'status': ['accepted_values']},), ['accepted_values_status', 'not_null_id', 'unique_id']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops.

```python
def test_names(config):
    out = []
    for col, tests in config.items():
        for t in tests:
            out.append(t + "_" + col)
    return sorted(out)
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def test_names(config):
    return sorted(f"{t}_{c}" for c, ts in config.items() for t in ts)
```

**Cost:** O(t log t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(t) time and O(t) space."

"I can do better: A sorted comprehension. That's O(t log t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/generate-dbt-tests)

<a id="q10"></a>
### Q10. Build Order of Models

🟡 Interview standard · *dbt run*

models maps a name to its SQL. Return a build order respecting ref() (alphabetical among those ready).

**Examples**

- `models = {'c': "{{ ref('a') }} {{ ref('b') }}", 'b': "{{ ref('a') }}", 'a': 'SELECT 1'}` → `['a', 'b', 'c']`

In [ ]:
def build_order(models):
    """
    models maps a name to its SQL. Return a build order respecting ref()
    (alphabetical among those ready).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(build_order, [
    (({'c': "{{ ref('a') }} {{ ref('b') }}", 'b': "{{ ref('a') }}", 'a': 'SELECT 1'},), ['a', 'b', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly scan for models whose references are built.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm with a heap.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeatedly scan for models whose references are built.

```python
def build_order(models):
    deps = {m: set(re.findall(r"ref\('(\w+)'\)", s)) for m, s in models.items()}
    done = []
    while len(done) < len(models):
        ready = sorted(m for m in models if m not in done and deps[m] <= set(done))
        done.append(ready[0])
    return done
```

**Cost:** O(m²) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's algorithm with a heap.

```python
def build_order(models):
    after, waiting = defaultdict(list), {}
    for m, s in models.items():
        refs = set(re.findall(r"ref\('(\w+)'\)", s))
        waiting[m] = len(refs)
        for r in refs:
            after[r].append(m)
    heap = sorted(m for m, n in waiting.items() if n == 0)
    order = []
    while heap:
        m = heapq.heappop(heap)
        order.append(m)
        for x in after[m]:
            waiting[x] -= 1
            if not waiting[x]:
                heapq.heappush(heap, x)
    return order
```

**Cost:** O(m log m + e) time, O(m + e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly scan for models whose references are built. That costs O(m²) time and O(m) space."

"I can do better: Kahn's algorithm with a heap. That's O(m log m + e) time and O(m + e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/build-order-of-models)

<a id="q11"></a>
### Q11. A Mart Built on Staging

🟡 Interview standard · *Layers of models*

Build daily revenue from a staging view. Return (day, total), by day.

**Examples**

- `` → `[['d1', 10], ['d2', 7], ['d3', 1]]`

In [ ]:
def daily_revenue():
    """
    Build daily revenue from a staging view. Return (day, total), by day.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(daily_revenue, [
    ((), [['d1', 10], ['d2', 7], ['d3', 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A CTE, all in one query.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two views, stacked, so the cleaning is reused.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A CTE, all in one query.

```python
def daily_revenue():
    return run_sql("""WITH s AS (SELECT DISTINCT id, day, amount FROM raw_orders WHERE status = 'paid')
        SELECT day, SUM(amount) FROM s GROUP BY day ORDER BY day""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two views, stacked, so the cleaning is reused.

```python
def daily_revenue():
    return run_sql("""CREATE VIEW stg AS SELECT DISTINCT id, day, amount FROM raw_orders WHERE status = 'paid';
        CREATE VIEW mart AS SELECT day, SUM(amount) AS t FROM stg GROUP BY day;
        SELECT * FROM mart ORDER BY day""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A CTE, all in one query. That costs O(n) time and O(d) space."

"I can do better: Two views, stacked, so the cleaning is reused. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/a-mart-built-on-staging)

<a id="q12"></a>
### Q12. Incremental Model

🟡 Interview standard · *Only new days*

existing maps day to total. new\_rows are (day, amount). Reprocess days ≥ the max existing day, replacing them. Return the table sorted.

**Examples**

- `existing = {'d1': 10, 'd2': 5}, new_rows = [('d2', 3), ('d2', 4), ('d3', 1), ('d1', 99)]` → `{'d1': 10, 'd2': 7, 'd3': 1}`

In [ ]:
def incremental_model(existing, new_rows):
    """
    existing maps day to total. new_rows are (day, amount). Reprocess days ≥ the
    max existing day, replacing them. Return the table sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(incremental_model, [
    (({'d1': 10, 'd2': 5}, [('d2', 3), ('d2', 4), ('d3', 1), ('d1', 99)]), {'d1': 10, 'd2': 7, 'd3': 1}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A filter, then group.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def incremental_model(existing, new_rows):
    cutoff = max(existing)
    fresh = {}
    for day, amount in new_rows:
        if day >= cutoff:
            fresh[day] = fresh.get(day, 0) + amount
    out = dict(existing)
    out.update(fresh)
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A filter, then group.

```python
def incremental_model(existing, new_rows):
    cut, fresh = max(existing), defaultdict(int)
    for d, a in new_rows:
        if d >= cut:
            fresh[d] += a
    return dict(sorted((existing | fresh).items()))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(n) time and O(d) space."

"I can do better: A filter, then group. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/incremental-model)

<a id="q13"></a>
### Q13. dbt Snapshot

🟡 Interview standard · *SCD 2 from a table*

Compare current source {id: value} with open snapshot rows (id, value, from). Close changed rows and open new ones at time t. Return (rows closed, rows opened), sorted.

**Examples**

- `open_rows = [(1, 'a', 0), (2, 'b', 0)], source = {1: 'a', 2: 'B', 3: 'c'}, t = 5` → `([2], [[2, 'B', 5], [3, 'c', 5]])`

In [ ]:
def snapshot_step(open_rows, source, t):
    """
    Compare current source {id: value} with open snapshot rows (id, value,
    from). Close changed rows and open new ones at time t. Return (rows closed,
    rows opened), sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(snapshot_step, [
    (([(1, 'a', 0), (2, 'b', 0)], {1: 'a', 2: 'B', 3: 'c'}, 5), ([2], [[2, 'B', 5], [3, 'c', 5]])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops over both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops over both.

```python
def snapshot_step(open_rows, source, t):
    closed, opened = [], []
    for i, v, _ in open_rows:
        if i in source and source[i] != v:
            closed.append(i)
            opened.append([i, source[i], t])
    known = [r[0] for r in open_rows]
    for i in source:
        if i not in known:
            opened.append([i, source[i], t])
    return (sorted(closed), sorted(opened))
```

**Cost:** O(n·m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary.

```python
def snapshot_step(open_rows, source, t):
    cur = {i: v for i, v, _ in open_rows}
    changed = [i for i, v in cur.items() if i in source and source[i] != v]
    new = [i for i in source if i not in cur]
    return (sorted(changed), sorted([i, source[i], t] for i in changed + new))
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops over both. That costs O(n·m) time and O(n) space."

"I can do better: A dictionary. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/dbt-snapshot)

<a id="q14"></a>
### Q14. dbt run --select

🟡 Interview standard · *Run part of the graph*

Given parent → children edges, return models selected by 'x+' (x and downstream), '+x' (x and upstream) or 'x', sorted.

**Examples**

- `graph = {'a': ['b'], 'b': ['c'], 'd': ['c']}, selector = 'b+'` → `['b', 'c']`
- `graph = {'a': ['b'], 'b': ['c'], 'd': ['c']}, selector = '+c'` → `['a', 'b', 'c', 'd']`
- `graph = {'a': ['b']}, selector = 'a'` → `['a']`

In [ ]:
def select_models(graph, selector):
    """
    Given parent → children edges, return models selected by 'x+' (x and
    downstream), '+x' (x and upstream) or 'x', sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(select_models, [
    (({'a': ['b'], 'b': ['c'], 'd': ['c']}, 'b+'), ['b', 'c']),
    (({'a': ['b'], 'b': ['c'], 'd': ['c']}, '+c'), ['a', 'b', 'c', 'd']),
    (({'a': ['b']}, 'a'), ['a']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeat scanning until nothing changes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A graph search in the right direction.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeat scanning until nothing changes.

```python
def select_models(graph, selector):
    name = selector.strip("+")
    out = {name}
    changed = True
    while changed:
        changed = False
        for p, kids in graph.items():
            for k in kids:
                if selector.endswith("+") and p in out and k not in out:
                    out.add(k)
                    changed = True
                if selector.startswith("+") and k in out and p not in out:
                    out.add(p)
                    changed = True
    return sorted(out)
```

**Cost:** O(V·E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A graph search in the right direction.

```python
def select_models(graph, selector):
    down, up = defaultdict(list), defaultdict(list)
    for p, kids in graph.items():
        for k in kids:
            down[p].append(k)
            up[k].append(p)
    name = selector.strip("+")
    links = down if selector.endswith("+") else up if selector.startswith("+") else {}
    seen, stack = set(), [name]
    while stack:
        n = stack.pop()
        if n not in seen:
            seen.add(n)
            stack += links.get(n, [])
    return sorted(seen)
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeat scanning until nothing changes. That costs O(V·E) time and O(V) space."

"I can do better: A graph search in the right direction. That's O(V + E) time and O(V) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/dbt-run-select)

<a id="q15"></a>
### Q15. View, Table or Materialized View?

🟡 Interview standard · *Pay at read or at write*

A query costs c to run. A view runs it on every read. A table costs c × rebuilds per day. Return the daily cost of each and the cheaper name (a tie goes to 'view').

**Examples**

- `c = 10, reads = 100, rebuilds = 1` → `(1000, 10, 'table')`
- `c = 10, reads = 1, rebuilds = 24` → `(10, 240, 'view')`

In [ ]:
def materialization(c, reads, rebuilds):
    """
    A query costs c to run. A view runs it on every read. A table costs c ×
    rebuilds per day. Return the daily cost of each and the cheaper name (a tie
    goes to 'view').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(materialization, [
    ((10, 100, 1), (1000, 10, 'table')),
    ((10, 1, 24), (10, 240, 'view')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Multiply and compare.

```python
def materialization(c, reads, rebuilds):
    view = c * reads
    table = c * rebuilds
    if view <= table:
        return (view, table, "view")
    return (view, table, "table")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def materialization(c, reads, rebuilds):
    return (c * reads, c * rebuilds, "view" if reads <= rebuilds else "table")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply and compare. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/view-table-or-materialized-view)

<a id="q16"></a>
### Q16. Warehouse Size vs Time

🟡 Interview standard · *Pay for compute time*

Each size up doubles credits per hour. With perfect scaling, it halves time, so cost stays equal; with an imperfect scale factor it costs more. Return (hours, credits) per size step, rounded to 2 decimals.

**Examples**

- `hours = 8, credits = 1, steps = [0, 1, 2], speedup = 2.0` → `[[8.0, 8.0], [4.0, 8.0], [2.0, 8.0]]`
- `hours = 8, credits = 1, steps = [0, 1, 2], speedup = 1.5` → `[[8.0, 8.0], [5.33, 10.67], [3.56, 14.22]]`

In [ ]:
def size_tradeoff(hours, credits, steps, speedup):
    """
    Each size up doubles credits per hour. With perfect scaling, it halves time,
    so cost stays equal; with an imperfect scale factor it costs more. Return
    (hours, credits) per size step, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(size_tradeoff, [
    ((8, 1, [0, 1, 2], 2.0), [[8.0, 8.0], [4.0, 8.0], [2.0, 8.0]]),
    ((8, 1, [0, 1, 2], 1.5), [[8.0, 8.0], [5.33, 10.67], [3.56, 14.22]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with powers.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with powers.

```python
def size_tradeoff(hours, credits, steps, speedup):
    out = []
    for s in steps:
        h = hours / speedup ** s
        out.append([round(h, 2), round(h * credits * 2 ** s, 2)])
    return out
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def size_tradeoff(hours, credits, steps, speedup):
    return [[round(hours / speedup ** s, 2), round(hours / speedup ** s * credits * 2 ** s, 2)] for s in steps]
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with powers. That costs O(s) time and O(s) space."

"I can do better: A comprehension. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/warehouse-size-vs-time)

<a id="q17"></a>
### Q17. Choose a Distribution Key

🟡 Interview standard · *MPP warehouses*

Hash-distributing rows by a key to n nodes: return skew (largest ÷ average, rounded to 2 decimals) for each candidate column, and the best.

**Examples**

- `columns = {'country': ['in', 'in', 'in', 'in', 'in', 'in', 'in', 'in', 'us', 'us'], 'user': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}, n = 2` → `({'country': 1.6, 'user': 1.0}, 'user')`

In [ ]:
def dist_key(columns, n):
    """
    Hash-distributing rows by a key to n nodes: return skew (largest ÷ average,
    rounded to 2 decimals) for each candidate column, and the best.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dist_key, [
    (({'country': ['in', 'in', 'in', 'in', 'in', 'in', 'in', 'in', 'us', 'us'], 'user': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}, 2), ({'country': 1.6, 'user': 1.0}, 'user')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper with `Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count in loops.

```python
def dist_key(columns, n):
    out = {}
    for name, values in columns.items():
        groups = {}
        for v in values:
            distinct = sorted(set(map(str, values)))
            node = distinct.index(str(v)) % n
            groups[node] = groups.get(node, 0) + 1
        out[name] = round(max(groups.values()) / (len(values) / n), 2)
    return (out, min(out, key=out.get))
```

**Cost:** O(c·r) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper with `Counter`.

```python
def dist_key(columns, n):
    from collections import Counter

    def skew(values):
        order = {v: i for i, v in enumerate(sorted(set(map(str, values))))}
        return round(max(Counter(order[str(v)] % n for v in values).values()) * n / len(values), 2)
    out = {k: skew(v) for k, v in columns.items()}
    return (out, min(out, key=out.get))
```

**Cost:** O(c·r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in loops. That costs O(c·r) time and O(n) space."

"I can do better: A helper with `Counter`. That's O(c·r) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `columns = {'country': ['in', 'in', 'in', 'in', 'in', 'in', 'in', 'in', 'us', 'us'], 'user': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}, n = 2`

1. skew() returns 1.6
2. skew() returns 1.0
3. dist\_key() returns ({'country': 1.6, 'user': 1.0}, 'user')

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/choose-a-distribution-key)

<a id="q18"></a>
### Q18. Sort Key Pruning

🟡 Interview standard · *Zone maps*

Blocks store min and max. Return blocks read for a range filter on the given column. Blocks are (min, max).

**Examples**

- `blocks = [(1, 10), (11, 20), (21, 30), (31, 40)], lo = 12, hi = 25` → `2`

In [ ]:
def blocks_read(blocks, lo, hi):
    """
    Blocks store min and max. Return blocks read for a range filter on the given
    column. Blocks are (min, max).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(blocks_read, [
    (([(1, 10), (11, 20), (21, 30), (31, 40)], 12, 25), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def blocks_read(blocks, lo, hi):
    n = 0
    for a, b in blocks:
        if a <= hi and b >= lo:
            n += 1
    return n
```

**Cost:** O(b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A generator.

```python
def blocks_read(blocks, lo, hi):
    return sum(a <= hi and b >= lo for a, b in blocks)
```

**Cost:** O(b) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(b) time and O(1) space."

"I can do better: A generator. That's O(b) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/sort-key-pruning)

<a id="q19"></a>
### Q19. Queue Wait

🟡 Interview standard · *Concurrency limits*

Queries with durations start in order, with at most c at once. Return each query's wait.

**Examples**

- `durations = [10, 10, 10, 5], c = 2` → `[0, 0, 10, 10]`

In [ ]:
def queue_wait(durations, c):
    """
    Queries with durations start in order, with at most c at once. Return each
    query's wait.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(queue_wait, [
    (([10, 10, 10, 5], 2), [0, 0, 10, 10]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of free times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A heap.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of free times.

```python
def queue_wait(durations, c):
    free = [0] * c
    out = []
    for d in durations:
        i = free.index(min(free))
        out.append(free[i])
        free[i] += d
    return out
```

**Cost:** O(n·c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A heap.

```python
def queue_wait(durations, c):
    heap, out = [0] * c, []
    for d in durations:
        start = heapq.heappop(heap)
        out.append(start)
        heapq.heappush(heap, start + d)
    return out
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of free times. That costs O(n·c) time and O(c) space."

"I can do better: A heap. That's O(n log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/queue-wait)

<a id="q20"></a>
### Q20. One Definition of a Metric

🟡 Interview standard · *The semantic layer*

Teams define revenue differently. Given definitions as filters on status, and rows (status, amount), return revenue per definition, and whether they all agree.

**Examples**

- `definitions = {'sales': ['paid', 'shipped'], 'finance': ['paid']}, rows = [('paid', 10), ('shipped', 5), ('failed', 2)]` → `({'sales': 15, 'finance': 10}, False)`

In [ ]:
def metric_versions(definitions, rows):
    """
    Teams define revenue differently. Given definitions as filters on status,
    and rows (status, amount), return revenue per definition, and whether they
    all agree.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(metric_versions, [
    (({'sales': ['paid', 'shipped'], 'finance': ['paid']}, [('paid', 10), ('shipped', 5), ('failed', 2)]), ({'sales': 15, 'finance': 10}, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops.

```python
def metric_versions(definitions, rows):
    out = {}
    for team, statuses in definitions.items():
        total = 0
        for s, a in rows:
            if s in statuses:
                total += a
        out[team] = total
    return (out, len(set(out.values())) == 1)
```

**Cost:** O(d·r) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def metric_versions(definitions, rows):
    out = {t: sum(a for s, a in rows if s in st) for t, st in definitions.items()}
    return (out, len(set(out.values())) == 1)
```

**Cost:** O(d·r) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(d·r) time and O(d) space."

"I can do better: A comprehension. That's O(d·r) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/warehouses-dbt/one-definition-of-a-metric)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does dbt do, and what doesn't it do?</b></summary>

It handles the T in ELT: dependency order, testing, documentation and versioning of SQL. It doesn't extract or load data, and it runs inside your warehouse.

</details>

<details>
<summary><b>2. How does an incremental model work?</b></summary>

The first run builds it all. Later runs filter to new data and merge on a unique key. Include a lookback for late data, and do an occasional full refresh.

</details>

<details>
<summary><b>3. Partitioning vs clustering in a warehouse?</b></summary>

Partitioning divides a table into coarse segments, usually by date. Clustering sorts data within them to improve block skipping.

</details>

<details>
<summary><b>4. How does Snowflake or BigQuery separate storage and compute?</b></summary>

Data lives in object storage, while independent compute clusters read it, so several workloads can scale and be billed separately.

</details>

<details>
<summary><b>5. View vs materialized view?</b></summary>

A view runs its query on each read. A materialized view stores results and refreshes them, trading storage and refresh cost for read speed.

</details>

<details>
<summary><b>6. Scenario: the warehouse bill doubled this month. What do you do?</b></summary>

Break it down by user, tag and query shape. Look for new full scans, warehouses that don't suspend, or repeated dashboards. Fix the top few first.

</details>

<details>
<summary><b>7. Scenario: an incremental model has drifted from the source. Why?</b></summary>

Late data outside the lookback, deletes that weren't handled, or logic changed without a full refresh. Compare with a full build, then widen the lookback or schedule refreshes.

</details>

<details>
<summary><b>8. Scenario: two dashboards show different revenue. How do you resolve it?</b></summary>

Find each definition and trace its lineage. Agree on one with the business owner, define it centrally, and retire the other.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: How a cloud warehouse works
- [ ] I can explain: Physical design: sort and distribution
- [ ] I can explain: dbt: software engineering for SQL
- [ ] I can explain: Incremental models, snapshots and cost control

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Warehouse Size vs Time
- [ ] [Q17](#q17) Choose a Distribution Key
- [ ] [Q18](#q18) Sort Key Pruning
- [ ] [Q19](#q19) Queue Wait
- [ ] [Q20](#q20) One Definition of a Metric

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.